<a href="https://colab.research.google.com/github/sun-mengwei/dtb-colab-experiments/blob/codex%2Fgame-dynamics-dtb/DTB_Game_Ver2/oscillatory_dtb_derministic_ver2_ci_diagnostics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Oscillatory deterministic game: DTB versus explicit Euler

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sun-mengwei/dtb-colab-experiments/blob/codex/game-dynamics-dtb/DTB_Game_Ver2/oscillatory_dtb_derministic_ver2_ci_diagnostics.ipynb)

\[
\Phi(x)=-\frac{\lambda}{2}(x_1^2+x_2^2)-\frac{\gamma}{2}(x_1-x_2)^2+
\frac{\varepsilon}{\omega}\{\cos(\omega x_1)+\cos(\omega x_2)\},\qquad \dot x=b(x)=\nabla\Phi(x).
\]
The map \(X(t;z)=T_\theta(z)\) is advanced in a frozen tangent chart:
\[
\alpha^k=\arg\min_\alpha\sum_i\|D_\theta T_\theta(z_i)\alpha-b(X_i^k)\|^2,\qquad
X^{k+1}=X^k+hD_\theta T_\theta\,\alpha^k.
\]
Every `REFIT_INTERVAL` steps, the accumulated tangent map is refitted into \(T_\theta\), after which the tangent bundle is recomputed. Direct explicit Euler with the same \(h\) is the reference, so the plotted discrepancy measures tangent-projection and refit error rather than high-order time-discretization error.

In [ ]:
from pathlib import Path
import os, subprocess, sys, math, time
import numpy as np
import matplotlib.pyplot as plt
import torch

REPO = Path("/content/dtb-colab-experiments")
BRANCH = "codex/game-dynamics-dtb"
if not (REPO / ".git").exists():
    subprocess.run(["git","clone","-q","--depth","1","--branch",BRANCH,
                    "https://github.com/sun-mengwei/dtb-colab-experiments.git",str(REPO)],check=True)
else:
    subprocess.run(["git","-C",str(REPO),"checkout","-q",BRANCH],check=True)
    subprocess.run(["git","-C",str(REPO),"pull","-q","--ff-only"],check=True)
EXPERIMENT_DIR = REPO / "DTB_Game_Ver2"
sys.path.insert(0, str(EXPERIMENT_DIR))
os.chdir(EXPERIMENT_DIR)

try:
    import torch._dynamo.compiled_autograd
except (AttributeError, ImportError):
    pass
from dtb import device, flat_params, jform_solve
from run_game_dtb import ResidualMLPMap, CurrentGameMap, game_dtb_basis_matrices, fit_map_to_target

LAMBDA,GAMMA,EPSILON,OMEGA = 0.5,0.2,0.5,4*math.pi
T,h = 1.0,1e-2
STEPS = round(T/h)
N_PROJ,N_EVAL = 256,512
WIDTH,DEPTH = 12,1
REFIT_INTERVAL = 10
FIT_STEPS,FIT_SAMPLES,FIT_BATCH,FIT_LR = 100,768,256,2e-3
CHUNK,RTOL,SOLVER = 256,1e-5,"svd_gpu"
DIRECTION_NORM_TOL = 1e-8
SEED,MODEL_SEED = 2026,91
DTYPE,DEVICE = torch.float32,device()
if DEVICE.type=="cuda": torch.set_float32_matmul_precision("high")
print({"device":str(DEVICE),"steps":STEPS,"h":h,"projection particles":N_PROJ,
       "evaluation particles":N_EVAL,"refit interval":REFIT_INTERVAL})

In [ ]:
def phi(x):
    x1,x2=x.unbind(-1)
    return (-.5*LAMBDA*(x1.square()+x2.square())-.5*GAMMA*(x1-x2).square()
            +(EPSILON/OMEGA)*(torch.cos(OMEGA*x1)+torch.cos(OMEGA*x2)))

def b(x):
    x1,x2=x.unbind(-1)
    return torch.stack((-LAMBDA*x1-GAMMA*(x1-x2)-EPSILON*torch.sin(OMEGA*x1),
                        -LAMBDA*x2-GAMMA*(x2-x1)-EPSILON*torch.sin(OMEGA*x2)),-1)

def uniform(n,g):
    return (2*torch.rand(n,2,generator=g,dtype=DTYPE)-1).to(DEVICE)

def euler(z):
    x=z.clone(); hist=[x.cpu()]
    for _ in range(STEPS):
        x=x+h*b(x); hist.append(x.cpu())
    return torch.stack(hist)

def particle_direction_diagnostics(projected_velocity,target_velocity,tol=DIRECTION_NORM_TOL):
    """Per-particle direction, vector-error, and magnitude diagnostics.

    The cosine is undefined when either vector is effectively zero, so those
    entries are NaN instead of being assigned an artificial direction.  The
    relative error and magnitude ratio only require a nonzero target velocity.
    """
    projected_norm=torch.linalg.vector_norm(projected_velocity,dim=-1)
    target_norm=torch.linalg.vector_norm(target_velocity,dim=-1)
    cosine_valid=(projected_norm>tol)&(target_norm>tol)
    target_valid=target_norm>tol

    cosine=torch.full_like(projected_norm,float("nan"))
    cosine[cosine_valid]=(
        (projected_velocity[cosine_valid]*target_velocity[cosine_valid]).sum(-1)
        /(projected_norm[cosine_valid]*target_norm[cosine_valid])
    ).clamp(-1.0,1.0)

    relative_error=torch.full_like(target_norm,float("nan"))
    magnitude_ratio=torch.full_like(target_norm,float("nan"))
    relative_error[target_valid]=(
        torch.linalg.vector_norm(
            projected_velocity[target_valid]-target_velocity[target_valid],dim=-1
        )/target_norm[target_valid]
    )
    magnitude_ratio[target_valid]=projected_norm[target_valid]/target_norm[target_valid]
    return cosine,relative_error,magnitude_ratio,cosine_valid

def dtb(z_eval):
    torch.manual_seed(MODEL_SEED)
    model=ResidualMLPMap(dim=2,width=WIDTH,depth=DEPTH,activation="tanh",dtype=DTYPE).to(DEVICE)
    theta0,structure,_=flat_params(model)
    sel=torch.arange(theta0.numel(),device=DEVICE)
    proj_gen=torch.Generator().manual_seed(SEED+1)

    def block():
        theta,_,_=flat_params(model)
        z=uniform(N_PROJ,proj_gen)
        yp,Jp,Jf=game_dtb_basis_matrices(theta,sel,z,model,structure,chunk=CHUNK)
        ye,Je,_=game_dtb_basis_matrices(theta,sel,z_eval,model,structure,chunk=CHUNK)
        return theta,yp.detach(),Jp.detach(),Jf.detach(),ye.detach(),Je.detach(),torch.zeros(sel.numel(),device=DEVICE)

    theta,yp,Jp,Jf,ye,Je,s=block()
    hist=[z_eval.cpu()]; residual=[]; refit_steps=[]; refit_rmse=[]
    alpha_norm=[]; sigma_min_retained=[]
    local={name:[] for name in (
        "cos_proj","relative_error_proj","magnitude_ratio_proj","valid_proj",
        "cos_eval","relative_error_eval","magnitude_ratio_eval","valid_eval",
    )}
    tic=time.perf_counter()
    for k in range(STEPS):
        # Both diagnostics are evaluated at the pre-update state X^k.
        xp=yp+h*torch.einsum("ndm,m->nd",Jp,s)
        xe=ye+h*torch.einsum("ndm,m->nd",Je,s)
        target_proj=b(xp)
        alpha=jform_solve(Jf,target_proj.reshape(-1),rtol=RTOL,method=SOLVER).detach()
        # Spectrum-only diagnostic: use the solver's same relative SVD cutoff.
        singular_values=torch.linalg.svdvals(Jf)
        retained=singular_values>RTOL*singular_values[0]
        alpha_norm.append(float(torch.linalg.vector_norm(alpha)))
        sigma_min_retained.append(
            float(singular_values[retained][-1]) if bool(retained.any()) else float("nan")
        )
        velocity_proj=torch.einsum("ndm,m->nd",Jp,alpha)
        velocity_eval=torch.einsum("ndm,m->nd",Je,alpha)
        target_eval=b(xe)

        proj_values=particle_direction_diagnostics(velocity_proj,target_proj)
        eval_values=particle_direction_diagnostics(velocity_eval,target_eval)
        for name,value in zip(
            ("cos_proj","relative_error_proj","magnitude_ratio_proj","valid_proj"),
            proj_values,
        ):
            local[name].append(value.detach().cpu())
        for name,value in zip(
            ("cos_eval","relative_error_eval","magnitude_ratio_eval","valid_eval"),
            eval_values,
        ):
            local[name].append(value.detach().cpu())

        residual.append(float(
            torch.linalg.norm(velocity_proj-target_proj)
            /(torch.linalg.norm(target_proj)+1e-30)
        ))
        s=s+alpha
        hist.append((ye+h*torch.einsum("ndm,m->nd",Je,s)).cpu())

        if (k+1)%REFIT_INTERVAL==0:
            target_map=CurrentGameMap(theta,sel,s,h,model,structure,chunk=CHUNK)
            rmse=fit_map_to_target(model,target_map,dim=2,low=-1,high=1,n_samples=FIT_SAMPLES,
                                   steps=FIT_STEPS,lr=FIT_LR,batch_size=FIT_BATCH)
            refit_steps.append(k+1); refit_rmse.append(rmse)
            hist[-1]=model(z_eval).detach().cpu()
            if k+1<STEPS: theta,yp,Jp,Jf,ye,Je,s=block()

        if (k+1)%20==0:
            print(f"{k+1:3d}/{STEPS}: residual={residual[-1]:.2e}, refits={len(refit_steps)}")
    print(f"DTB wall time: {(time.perf_counter()-tic)/60:.2f} min")
    local={name:torch.stack(values) for name,values in local.items()}
    return (torch.stack(hist),np.asarray(residual),np.asarray(refit_steps),
            np.asarray(refit_rmse),local,np.asarray(alpha_norm),
            np.asarray(sigma_min_retained))

# Two exact implementation checks.
q=torch.tensor([[.17,-.31]],device=DEVICE,requires_grad=True)
assert (torch.autograd.grad(phi(q).sum(),q)[0]-b(q)).abs().max()<2e-6
torch.manual_seed(MODEL_SEED)
m=ResidualMLPMap(dim=2,width=WIDTH,depth=DEPTH,activation="tanh",dtype=DTYPE).to(DEVICE)
assert (m(q.detach())-q.detach()).abs().max()==0

# Check direction handling without hiding zero-velocity cases.
u_check=torch.tensor([[1.,0.],[0.,0.]],device=DEVICE)
b_check=torch.tensor([[1.,0.],[1.,0.]],device=DEVICE)
c_check,_,_,valid_check=particle_direction_diagnostics(u_check,b_check)
assert torch.allclose(c_check[:1],torch.ones(1,device=DEVICE))
assert torch.isnan(c_check[1]) and not bool(valid_check[1])


In [ ]:
g=torch.Generator().manual_seed(SEED)
z_eval=uniform(N_EVAL,g)
X_euler=euler(z_eval)
(X_dtb,residual,refit_steps,refit_rmse,local_direction,
 alpha_norm,sigma_min_retained)=dtb(z_eval)

times=np.arange(STEPS+1)*h
rmse=torch.sqrt(torch.mean(torch.sum((X_dtb-X_euler).square(),-1),-1))
rel=rmse/(torch.sqrt(torch.mean(torch.sum(X_euler.square(),-1),-1))+1e-12)
P_euler=phi(X_euler).mean(1); P_dtb=phi(X_dtb).mean(1)

print(f"final RMSE={rmse[-1]:.3e}, final relative RMSE={rel[-1]:.3e}")
print(f"median/max projection residual={np.median(residual):.3e}/{residual.max():.3e}")
print("refit RMSE:",np.array2string(refit_rmse,precision=3))

snap=np.linspace(0,STEPS,5,dtype=int)
a=torch.linspace(-1.05,1.05,150)
x1,x2=torch.meshgrid(a,a,indexing="xy")
grid=phi(torch.stack((x1.ravel(),x2.ravel()),-1)).reshape(x1.shape).numpy()
fig,ax=plt.subplots(2,5,figsize=(15,6),sharex=True,sharey=True)
for r,(H,name) in enumerate(((X_euler,"Euler"),(X_dtb,"DTB"))):
    for j,k in enumerate(snap):
        p=H[k].numpy(); ax[r,j].contour(x1,x2,grid,levels=16,linewidths=.45)
        ax[r,j].scatter(p[:,0],p[:,1],s=6,alpha=.5)
        ax[r,j].set(title=f"{name}, t={k*h:.2f}",xlim=(-1.05,1.05),ylim=(-1.05,1.05))
        ax[r,j].set_aspect("equal"); ax[r,j].grid(alpha=.15)
plt.tight_layout(); plt.show()

fig,ax=plt.subplots(1,3,figsize=(15,4))
ax[0].semilogy(times,rmse+1e-12,label="absolute"); ax[0].semilogy(times,rel+1e-12,label="relative")
ax[0].set_title("DTB error against Euler"); ax[0].legend()
ax[1].semilogy(times[1:],residual+1e-12); ax[1].set_title(r"$\|J\alpha-b\|/\|b\|$")
ax[2].plot(times,P_euler,label="Euler"); ax[2].plot(times,P_dtb,label="DTB")
ax[2].set_title("mean potential"); ax[2].legend()
for a0 in ax:
    a0.set_xlabel("time"); a0.grid(alpha=.25)
    for s in refit_steps: a0.axvline(s*h,ls="--",lw=.7,alpha=.35)
plt.tight_layout(); plt.show()

## Per-particle local-direction diagnostic

For each projection particle and each fixed held-out evaluation particle, the notebook now records

\[
c_i=\frac{(J_i\alpha)^\top b_i}{\lVert J_i\alpha\rVert\,\lVert b_i\rVert}.
\]

The projection particles are the in-sample points used to solve for \(\alpha\); the evaluation particles are fixed throughout the run and are the primary plotted diagnostic. If either vector norm is below `DIRECTION_NORM_TOL`, \(c_i\) is stored as `NaN` and excluded from fractions and quantiles. The per-particle relative vector error \(r_i\) and magnitude ratio \(\rho_i\) are retained in `local_direction` for later inspection, but are not plotted yet.

To keep the investigation focused, the local-direction plots remain limited to the held-out fractions and the global projection residual against the held-out 5th-percentile lower tail of \(c_i\). A final two-panel figure adds only the requested solver diagnostics: \(\lVert\alpha_k\rVert_2\) and \(\sigma_{\min,\mathrm{retained}}(J_k)\) over time.

In [ ]:
c_eval=local_direction["cos_eval"].numpy()
c_proj=local_direction["cos_proj"].numpy()
valid_eval=np.isfinite(c_eval)
valid_proj=np.isfinite(c_proj)
direction_times=times[:-1]  # c_i^k is evaluated at pre-update X^k.

def fraction_among_valid(condition,valid):
    numerator=np.sum(condition&valid,axis=1)
    denominator=np.sum(valid,axis=1)
    return np.divide(
        numerator,denominator,
        out=np.full(numerator.shape,np.nan,dtype=float),
        where=denominator>0,
    )

fraction_negative=fraction_among_valid(c_eval<0.0,valid_eval)
fraction_above_half=fraction_among_valid(c_eval>0.5,valid_eval)
lower_tail=np.nanquantile(c_eval,0.05,axis=1)
valid_fraction_eval=valid_eval.mean(axis=1)
valid_fraction_proj=valid_proj.mean(axis=1)

fig,ax=plt.subplots(figsize=(7,4))
ax.plot(direction_times,fraction_negative,label=r"fraction $c_i<0$")
ax.plot(direction_times,fraction_above_half,label=r"fraction $c_i>0.5$")
for index,step in enumerate(refit_steps):
    ax.axvline(step*h,ls="--",lw=.7,alpha=.3,label="refit" if index==0 else None)
ax.set(xlabel="time",ylabel="portion of valid evaluation particles",ylim=(-.02,1.02),
       title="Held-out local-direction alignment")
ax.grid(alpha=.25); ax.legend(); plt.tight_layout(); plt.show()

fig,ax=plt.subplots(figsize=(6,4.5))
points=ax.scatter(residual,lower_tail,c=direction_times,cmap="viridis",s=28)
ax.axhline(0,color="0.5",ls="--",lw=.8)
ax.set_xscale("log")
ax.set(xlabel=r"global projection residual $\|J\alpha-b\|/\|b\|$",
       ylabel=r"held-out 5th percentile of $c_i$",
       title="Global residual versus lower-tail direction")
ax.grid(alpha=.25); plt.colorbar(points,ax=ax,label="time")
plt.tight_layout(); plt.show()

print(f"minimum valid fraction: projection={np.nanmin(valid_fraction_proj):.3f}, "
      f"evaluation={np.nanmin(valid_fraction_eval):.3f}")
print(f"final held-out fractions: c_i<0: {fraction_negative[-1]:.3f}; "
      f"c_i>0.5: {fraction_above_half[-1]:.3f}; q05={lower_tail[-1]:.3f}")

fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].semilogy(direction_times,alpha_norm)
ax[0].set(xlabel="time",ylabel=r"$\|\alpha_k\|_2$",
          title=r"Coefficient norm $\|\alpha_k\|_2$")
ax[1].semilogy(direction_times,sigma_min_retained)
ax[1].set(xlabel="time",ylabel=r"$\sigma_{\min,\mathrm{retained}}(J_k)$",
          title="Smallest retained singular value")
for a0 in ax:
    a0.grid(alpha=.25)
    for step in refit_steps:
        a0.axvline(step*h,ls="--",lw=.7,alpha=.3)
plt.tight_layout(); plt.show()
